### Types of Missing Data in Pandas
Pandas represents missing values differently depending on dtype.

| Missing Type | Used For                                       |
| ------------ | ---------------------------------------------- |
| `NaN`        | Numeric data                                   |
| `None`       | Python object / mixed                          |
| `NaT`        | Datetime                                       |
| `<NA>`       | Nullable dtypes (`Int64`, `string`, `boolean`) |


In [13]:
import pandas as pd
import numpy as np

In [14]:
s = pd.Series([1, None, 3])
s

0    1.0
1    NaN
2    3.0
dtype: float64

### Detecting Missing Data

In [15]:
df = pd.DataFrame({
    "user_id": [301, 302, 303, 304, 305, 306, 307],
    "age": [24, None, 29, 35, None, 41, 27],
    "gender": ["M", "F", None, "M", "F", None, "M"],
    "city": ["Pune", "Mumbai", "Delhi", None, "Pune", "Bangalore", None],
    "account_balance": [12000.50, 8500.00, None, 15000.75, None, 20000.00, 9800.25],
    "last_login": [
        "2024-02-01",
        None,
        "2024-02-05",
        "invalid_date",
        "2024-02-10",
        None,
        "2024-02-12"
    ],
    "is_premium_user": [True, None, False, True, None, True, False]
})

In [16]:
df.isna()

,user_id,age,gender,city,account_balance,last_login,is_premium_user
0,False,False,False,False,False,False,False
1,False,True,False,False,False,True,True
2,False,False,True,False,True,False,False
3,False,False,False,True,False,False,False
4,False,True,False,False,True,False,True
5,False,False,True,False,False,True,False
6,False,False,False,True,False,False,False


In [17]:
df.notna()

,user_id,age,gender,city,account_balance,last_login,is_premium_user
0,True,True,True,True,True,True,True
1,True,False,True,True,True,False,False
2,True,True,False,True,False,True,True
3,True,True,True,False,True,True,True
4,True,False,True,True,False,True,False
5,True,True,False,True,True,False,True
6,True,True,True,False,True,True,True


In [18]:
df.isna().sum()

user_id            0
age                2
gender             2
city               2
account_balance    2
last_login         2
is_premium_user    2
dtype: int64

### Missing Data Patterns
| Scenario           | Meaning               |
| ------------------ | --------------------- |
| User skipped field | Safe to fill          |
| Data not collected | Might be okay         |
| System failure     | Dangerous             |
| Business logic     | Missing = information |


Example:
- Missing `city` -> fine
- Missing `last_login` -> meaningful
- Missing `salary` -> risky

### Dropping Missing Data

#### Drop rows with any missing value
```python
df.dropna()
```

Dangerous - often removes too much data

#### Drop rows based on specific columns
```python
df.dropna(subset=["salary"])
```

Targeted & Safer

#### Drop columns with too many missing values
```python
df.dropna(axis=1, thresh=3)
```
Keeps columns with at least 3 non-null values

### Filling Missing Data (`fillna`)
#### Fill with constant value

In [19]:
df["city"] = df["city"].fillna("Unknown")
df

,user_id,age,gender,city,account_balance,last_login,is_premium_user
0,301,24.0,M,Pune,12000.50,2024-02-01,True
1,302,NaN,F,Mumbai,8500.00,None,None
2,303,29.0,None,Delhi,NaN,2024-02-05,False
3,304,35.0,M,Unknown,15000.75,invalid_date,True
4,305,NaN,F,Pune,NaN,2024-02-10,None
5,306,41.0,None,Bangalore,20000.00,None,True
6,307,27.0,M,Unknown,9800.25,2024-02-12,False


#### Fill mean/median (Numeric)

In [20]:
df["age"] = df["age"].fillna(df["age"].median())
df

,user_id,age,gender,city,account_balance,last_login,is_premium_user
0,301,24.0,M,Pune,12000.50,2024-02-01,True
1,302,29.0,F,Mumbai,8500.00,None,None
2,303,29.0,None,Delhi,NaN,2024-02-05,False
3,304,35.0,M,Unknown,15000.75,invalid_date,True
4,305,29.0,F,Pune,NaN,2024-02-10,None
5,306,41.0,None,Bangalore,20000.00,None,True
6,307,27.0,M,Unknown,9800.25,2024-02-12,False


#### Forward/Backward Fill (Time-Series)
```python
df["price"] = df["price"].ffill()
df["price"] = df["price"].bfill()
```

1. `ffill()` **(forward fill)**
    - Fills missing values by carrying the last known value forward.
    - Example:
```
price: [10, NaN, NaN, 15]
after ffill → [10, 10, 10, 15]
```

2. `bfill()` **(backward fill)**
    - Fills remaining missing values using the next known value backward.
    - This mainly affects NaNs at the start of the column.
    - Example:
```
price: [NaN, NaN, 10]
after bfill → [10, 10, 10]
```

**Why use both?**

Using both ffill() then bfill() ensures:
- Gaps in the middle are filled using previous values
- Gaps at the beginning are filled using the first valid value

### Datetime Missing Values (`NaT`)
#### Safe datetime conversion
1. `pd.to_datetime(...)`
- Tries to parse each value in df["joined_on"] as a date/time.
- Accepts many formats automatically:
    - `"2024-01-15"`
    - `"15/01/2024"`
    - `"Jan 15, 2024"`
    - `"2024-01-15 10:30"`

2. `errors="coerce"`
- If a value cannot be converted, pandas:
    - does not raise an error
    - replaces it with `NaT` (Not a Time)

In [21]:
df["last_login"] = pd.to_datetime(df["last_login"], errors='coerce')
df

,user_id,age,gender,city,account_balance,last_login,is_premium_user
0,301,24.0,M,Pune,12000.50,2024-02-01,True
1,302,29.0,F,Mumbai,8500.00,NaT,None
2,303,29.0,None,Delhi,NaN,2024-02-05,False
3,304,35.0,M,Unknown,15000.75,NaT,True
4,305,29.0,F,Pune,NaN,2024-02-10,None
5,306,41.0,None,Bangalore,20000.00,NaT,True
6,307,27.0,M,Unknown,9800.25,2024-02-12,False


### Nullable dtypes & Missing Values 
#### Classic Pandas (Bad)
```python
pd.Series([1, None, 3]) #float64
```

#### Production Pandas (Good)
```python
pd.Series([1, None, 3], dtype="Int64")
```

### Conditional Filling

In [22]:
df.loc[df["age"].isna(), "age"] = df["age"].median()

| Column Type         | Strategy         |
| ------------------- | ---------------- |
| Numeric feature     | Median           |
| Categorical feature | `"Unknown"`      |
| Target variable     | Never fill     |
| Timestamp           | Keep `NaT`       |
| Boolean             | Nullable boolean |


### Quick Summary Table
| Task              | Function                       |
| ----------------- | ------------------------------ |
| Detect missing    | `isna()`                       |
| Count missing     | `isna().sum()`                 |
| Drop rows         | `dropna()`                     |
| Fill values       | `fillna()`                     |
| Time-safe parsing | `to_datetime(errors="coerce")` |
| Production safety | Nullable dtypes                |


### Exercise 1
`User transaction data` from an upstream service:
```python
import pandas as pd

df = pd.DataFrame({
    "user_id": [201, 202, 203, 204, 205, 206],
    "age": [25, None, 30, None, 28, 35],
    "city": ["Pune", "Mumbai", None, "Delhi", None, "Pune"],
    "last_login": ["2024-01-10", None, "2024-01-12", "invalid_date", "2024-01-15", None],
    "spend": [1200, 1500, None, 800, 950, None]
})
```
- Show `count of missing values per column`
- Identify `which columns are safe to fill` and which are `dangerous`


In [23]:
import pandas as pd

df = pd.DataFrame({
    "user_id": [201, 202, 203, 204, 205, 206],
    "age": [25, None, 30, None, 28, 35],
    "city": ["Pune", "Mumbai", None, "Delhi", None, "Pune"],
    "last_login": ["2024-01-10", None, "2024-01-12", "invalid_date", "2024-01-15", None],
    "spend": [1200, 1500, None, 800, 950, None]
})

df.isna()

,user_id,age,city,last_login,spend
0,False,False,False,False,False
1,False,True,False,True,False
2,False,False,True,False,True
3,False,True,False,False,False
4,False,False,True,False,False
5,False,False,False,True,True


In [24]:
df.isna().sum()

user_id       0
age           2
city          2
last_login    2
spend         2
dtype: int64

|Column|Missing?|Safe to Fill?|Reason
|-------|--------|------------|---------|
|`age`|Yes|Yes|Demographic, can be imputed|
|`city`|Yes|Yes|Categorical, safe placeholder|
|`last_login`|Yes|No|Logic sensitive|
|`spend`|Yes|No|ML target/feature sensitive|

### Exercise 2
1. Convert last_login to datetime
2. Detect invalid / missing dates
3. Decide:
    - Drop rows?
    - Fill dates?
    - Keep as NaT?

In [25]:
df.dtypes

user_id         int64
age           float64
city           object
last_login     object
spend         float64
dtype: object

In [26]:
df["last_login"] = pd.to_datetime(df["last_login"], errors="coerce")
df.dtypes

user_id                int64
age                  float64
city                  object
last_login    datetime64[ns]
spend                float64
dtype: object

- Invalid dates -> `NaT`
- Strings -> real timestamps

In [27]:
df["last_login"].isna().sum()

np.int64(3)

**Decision:**

**Keep missing as `NaT`**

Why:
- Filling timestamps introduces **fake activity**
- ML features like "days since last login" would be corrupted
- APIs can be explicitly return `null`

### Exercise 3
For columns:
- `age`
- `spend`
1. Decide whether to:
    - Fill with mean / median
    - Fill with constant
    - Leave missing
2. Implement the chosen strategy
3. Explain `why this is safe`

**Column:** `age`
- Demographic data
- Median is robust to outliers

In [28]:
df["age"] = df["age"].fillna(df["age"].median())
df["age"]

0    25.0
1    29.0
2    30.0
3    29.0
4    28.0
5    35.0
Name: age, dtype: float64

**Column:** `spend`
- Monetary data
- Mean is skew-sensitive
- Zero has semantic meaning (user didn't spend)

**Decision:** Fill with `0`

Because missing spend usually means **no recorded spend**

In [29]:
df["spend"] = df["spend"].fillna(0)
df

,user_id,age,city,last_login,spend
0,201,25.0,Pune,2024-01-10,1200.0
1,202,29.0,Mumbai,NaT,1500.0
2,203,30.0,None,2024-01-12,0.0
3,204,29.0,Delhi,NaT,800.0
4,205,28.0,None,2024-01-15,950.0
5,206,35.0,Pune,NaT,0.0


### Exercise 4
For column:
- `city`
1. Decide:
    - Fill with `"Unknown"`
    - Drop rows
    - Infer from other data
2. Implement your choice
3. Explain trade-offs

**Column:** `city`
- Categorical
- Missing likely due to incomplete profile

**Decision:** Fill with `Unknown`

In [30]:
df["city"] = df["city"].fillna("Unknown")
df["city"]

0       Pune
1     Mumbai
2    Unknown
3      Delhi
4    Unknown
5       Pune
Name: city, dtype: object

| Option      | Pros           | Cons                |
| ----------- | -------------- | ------------------- |
| `"Unknown"` | Safe, explicit | Adds category       |
| Drop rows   | Clean          | Data loss           |
| Infer       | Fancy          | Risky & wrong often |


### Exercise 5
After all operations:
1. Verify no unexpected missing values
2. Print:
   - `df.info()`
   - `df.isna().sum()`

In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   user_id     6 non-null      int64         
 1   age         6 non-null      float64       
 2   city        6 non-null      object        
 3   last_login  3 non-null      datetime64[ns]
 4   spend       6 non-null      float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(1)
memory usage: 372.0+ bytes


In [32]:
df.isna().sum()

user_id       0
age           0
city          0
last_login    3
spend         0
dtype: int64

In [33]:
df

,user_id,age,city,last_login,spend
0,201,25.0,Pune,2024-01-10,1200.0
1,202,29.0,Mumbai,NaT,1500.0
2,203,30.0,Unknown,2024-01-12,0.0
3,204,29.0,Delhi,NaT,800.0
4,205,28.0,Unknown,2024-01-15,950.0
5,206,35.0,Pune,NaT,0.0


### When NOT to Fill Missing Values

Do **NOT** fill missing values when:
- The value represents `time` (timestamps)
- The value is a `target variable`
- Missing itself has `semantic meaning`
- Filling would introduce `false information`